# DistilBERT Intent Classifier

This notebook trains and evaluates a DistilBERT-based intent classifier
for the ResolveAI customer-support dataset.

The model is trained using the corrected leakage-safe generic-placeholder
training split and evaluated on the corresponding validation split.

The official test split remains untouched during model development.

After model selection, the frozen model will also be evaluated on the
independent OOD challenge benchmark to measure generalization to more
realistic customer language.

In [1]:
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Using device: cpu


### Model Configuration

In [4]:
MODEL_NAME = "distilbert-base-uncased"

TEXT_COLUMN = "processed_instruction"
TARGET_COLUMN = "intent"

### Load train and validation

In [5]:
train_df = pd.read_csv(
    "../data/processed/generic/train.csv"
)

validation_df = pd.read_csv(
    "../data/processed/generic/validation.csv"
)

print(
    "Train shape:",
    train_df.shape
)

print(
    "Validation shape:",
    validation_df.shape
)

Train shape: (18780, 9)
Validation shape: (4016, 9)


#### Verifying required columns

In [6]:
print(
    train_df.columns.tolist()
)

['flags', 'instruction', 'category', 'intent', 'response', 'normalized_instruction', 'split_group_key', 'processed_instruction', 'split']


In [7]:
required_columns = {
    TEXT_COLUMN,
    TARGET_COLUMN,
}

assert required_columns.issubset(
    train_df.columns
)

assert required_columns.issubset(
    validation_df.columns
)

print(
    "Required columns found."
)

Required columns found.


In [8]:
print(
    train_df[
        [
            TEXT_COLUMN,
            TARGET_COLUMN
        ]
    ]
    .isna()
    .sum()
)

print()

print(
    validation_df[
        [
            TEXT_COLUMN,
            TARGET_COLUMN
        ]
    ]
    .isna()
    .sum()
)

processed_instruction    0
intent                   0
dtype: int64

processed_instruction    0
intent                   0
dtype: int64


#### Check intents

In [9]:
print(
    "Train intents:",
    train_df[TARGET_COLUMN].nunique()
)

print(
    "Validation intents:",
    validation_df[TARGET_COLUMN].nunique()
)

Train intents: 27
Validation intents: 27


In [10]:
sorted(
    train_df[TARGET_COLUMN].unique()
)

['cancel_order',
 'change_order',
 'change_shipping_address',
 'check_cancellation_fee',
 'check_invoice',
 'check_payment_methods',
 'check_refund_policy',
 'complaint',
 'contact_customer_service',
 'contact_human_agent',
 'create_account',
 'delete_account',
 'delivery_options',
 'delivery_period',
 'edit_account',
 'get_invoice',
 'get_refund',
 'newsletter_subscription',
 'payment_issue',
 'place_order',
 'recover_password',
 'registration_problems',
 'review',
 'set_up_shipping_address',
 'switch_account',
 'track_order',
 'track_refund']

### Create ```label2id``` and ```id2label```

In [11]:
# Creating deterministic mapping

intent_labels = sorted(
  train_df[TARGET_COLUMN].unique()
)

label2id = {
  label: idx
  for idx, label in enumerate(
    intent_labels
  )
}

id2label = {
  idx: label
  for label, idx in label2id.items()
}

print(
  "Number of labels: ",
  len(label2id)
)

Number of labels:  27


In [12]:
label2id

{'cancel_order': 0,
 'change_order': 1,
 'change_shipping_address': 2,
 'check_cancellation_fee': 3,
 'check_invoice': 4,
 'check_payment_methods': 5,
 'check_refund_policy': 6,
 'complaint': 7,
 'contact_customer_service': 8,
 'contact_human_agent': 9,
 'create_account': 10,
 'delete_account': 11,
 'delivery_options': 12,
 'delivery_period': 13,
 'edit_account': 14,
 'get_invoice': 15,
 'get_refund': 16,
 'newsletter_subscription': 17,
 'payment_issue': 18,
 'place_order': 19,
 'recover_password': 20,
 'registration_problems': 21,
 'review': 22,
 'set_up_shipping_address': 23,
 'switch_account': 24,
 'track_order': 25,
 'track_refund': 26}

In [13]:
train_df["label"] = (
    train_df[TARGET_COLUMN]
    .map(label2id)
)

validation_df["label"] = (
    validation_df[TARGET_COLUMN]
    .map(label2id)
)

In [14]:
train_df[
    [
        TEXT_COLUMN,
        TARGET_COLUMN,
        "label"
    ]
].head()

,processed_instruction,intent,label
0,question about cancelling order ENTITY_TOKEN,cancel_order,0
1,i need help cancelling puchase ENTITY_TOKEN,cancel_order,0
2,I need to cancel purchase ENTITY_TOKEN,cancel_order,0
3,can you help me cancel order ENTITY_TOKEN?,cancel_order,0
4,"I can no longer afford order ENTITY_TOKEN, can...",cancel_order,0


In [15]:
assert train_df["label"].notna().all()
assert validation_df["label"].notna().all()

assert train_df["label"].nunique() == 27
assert validation_df["label"].nunique() == 27

print(
    "Label encoding validated."
)

Label encoding validated.


## Convert Pandas DataFrames to Hugging Face Datasets

In [16]:
train_dataset = Dataset.from_pandas(
    train_df[
        [
            TEXT_COLUMN,
            "label"
        ]
    ],
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df[
        [
            TEXT_COLUMN,
            "label"
        ]
    ],
    preserve_index=False
)

In [17]:
print(train_dataset)
print(validation_dataset)

Dataset({
    features: ['processed_instruction', 'label'],
    num_rows: 18780
})
Dataset({
    features: ['processed_instruction', 'label'],
    num_rows: 4016
})


In [18]:
train_dataset[0]

{'processed_instruction': 'question about cancelling order ENTITY_TOKEN',
 'label': 0}

## Load DistilBERT Tokenizer

In [20]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [21]:
# Inspecting the tokenizer

sample_text = (
    train_df[TEXT_COLUMN]
    .iloc[0]
)

print(sample_text)

question about cancelling order ENTITY_TOKEN


In [22]:
sample_tokens = tokenizer.tokenize(
    sample_text
)

sample_tokens

['question', 'about', 'cancel', '##ling', 'order', 'entity', '_', 'token']

#### Convert tokens to numerical IDs

In [23]:
sample_encoding = tokenizer(
    sample_text
)

sample_encoding

{'input_ids': [101, 3160, 2055, 17542, 2989, 2344, 9178, 1035, 19204, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [24]:
# Decoding the IDs

tokenizer.convert_ids_to_tokens(
    sample_encoding["input_ids"]
)

['[CLS]',
 'question',
 'about',
 'cancel',
 '##ling',
 'order',
 'entity',
 '_',
 'token',
 '[SEP]']

### Check sequence lengths

In [25]:
token_lengths = (
    train_df[TEXT_COLUMN]
    .apply(
        lambda text: len(
            tokenizer.encode(
                text,
                add_special_tokens=True
            )
        )
    )
)

In [26]:
token_lengths.describe(
    percentiles=[
        0.90,
        0.95,
        0.99
    ]
)

count    18780.000000
mean        12.215016
std          3.190224
min          3.000000
90%         16.000000
95%         18.000000
99%         20.000000
max         26.000000
Name: processed_instruction, dtype: float64

In [27]:
print(
    "Maximum token length:",
    token_lengths.max()
)

Maximum token length: 26


### Tokenization function definition

In [28]:
MAX_LENGTH = 64

In [29]:
def tokenize_batch(batch):

    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        max_length=MAX_LENGTH
    )

In [30]:
tokenized_train = (
    train_dataset.map(
        tokenize_batch,
        batched=True
    )
)

tokenized_validation = (
    validation_dataset.map(
        tokenize_batch,
        batched=True
    )
)

Map:   0%|          | 0/18780 [00:00<?, ? examples/s]

Map:   0%|          | 0/4016 [00:00<?, ? examples/s]

In [31]:
tokenized_train[0]

{'processed_instruction': 'question about cancelling order ENTITY_TOKEN',
 'label': 0,
 'input_ids': [101, 3160, 2055, 17542, 2989, 2344, 9178, 1035, 19204, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

## Dynamic Padding

In [32]:
from transformers import (
    DataCollatorWithPadding,
)

In [33]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

## Load DistilBERT for 27-Class Classification

In [34]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(intent_labels),
    id2label=id2label,
    label2id=label2id
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [35]:
print(
    "Number of labels:",
    model.config.num_labels
)

print(
    "Model labels:",
    model.config.id2label
)

Number of labels: 27
Model labels: {0: 'cancel_order', 1: 'change_order', 2: 'change_shipping_address', 3: 'check_cancellation_fee', 4: 'check_invoice', 5: 'check_payment_methods', 6: 'check_refund_policy', 7: 'complaint', 8: 'contact_customer_service', 9: 'contact_human_agent', 10: 'create_account', 11: 'delete_account', 12: 'delivery_options', 13: 'delivery_period', 14: 'edit_account', 15: 'get_invoice', 16: 'get_refund', 17: 'newsletter_subscription', 18: 'payment_issue', 19: 'place_order', 20: 'recover_password', 21: 'registration_problems', 22: 'review', 23: 'set_up_shipping_address', 24: 'switch_account', 25: 'track_order', 26: 'track_refund'}


## Define Evaluation Metrics

In [36]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            predictions,
            average="macro",
            zero_division=0
        )
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }

## CPU Training Smoke Test

In [37]:
smoke_train_dataset = (
    tokenized_train
    .shuffle(seed=SEED)
    .select(range(1000))
)

smoke_validation_dataset = (
    tokenized_validation
    .shuffle(seed=SEED)
    .select(range(500))
)

In [38]:
SMOKE_OUTPUT_DIR = (
    "../models/distilbert_smoke_test"
)

In [39]:
smoke_training_args = TrainingArguments(
    output_dir=SMOKE_OUTPUT_DIR,

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="no",

    logging_steps=25,

    report_to="none",

    seed=SEED
)

In [41]:
smoke_trainer = Trainer(
    model=model,
    args=smoke_training_args,

    train_dataset=smoke_train_dataset,
    eval_dataset=smoke_validation_dataset,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics
)

### Timed Smoke Test

In [42]:
import time

start_time = time.time()

smoke_train_result = (
    smoke_trainer.train()
)

elapsed_time = (
    time.time()
    - start_time
)

print(
    f"Smoke-test training time: "
    f"{elapsed_time / 60:.2f} minutes"
)

c:\D_Drive\ResolveAI\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,3.020629,2.978513,0.254000,0.370269,0.242070,0.196324


Smoke-test training time: 0.72 minutes


In [43]:
smoke_metrics = (
    smoke_trainer.evaluate()
)

smoke_metrics

c:\D_Drive\ResolveAI\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Macro Precision,Macro Recall,Macro F1
3.020629,2.978513,1,0.254000,0.370269,0.242070,0.196324


{'eval_loss': 2.978513479232788,
 'eval_accuracy': 0.254,
 'eval_macro_precision': 0.3702687884870403,
 'eval_macro_recall': 0.24207013201631392,
 'eval_macro_f1': 0.19632399066981834}